# R2-01 · Datos: traer, cruzar y limpiar — Lección

**Bootcamp de Datos para Funcionarios Públicos · Formación Pública**
Línea B · *Data Scientist* · Semana 3 · Se ejecuta en Google Colab.

> 🚀 **Cómo se trabaja:** lee, ejecuta cada celda con **▶︎** (o `Shift`+`Enter`) y completa los `TODO`. Cada ejercicio termina en una **celda de chequeo** que muestra ✅ si está bien o una pista si todavía no.

---

## ¿Qué vas a lograr?

1. **Cruzar** tablas con `merge` y resumir con `groupby`.
2. Detectar y tratar **nulos**.
3. Eliminar **duplicados**.
4. Corregir **tipos** de columnas.

**Competencia de salida:** construir un dataset analítico limpio y listo para modelar.

**Dato real:** compras públicas (ChileCompra): monto, categoría, región, tamaño de proveedor.

**Entregable:** que las **4 celdas de chequeo** muestren ✅.

In [ ]:
# ── Preparación del entorno (ejecuta esta celda primero) ──────────────────────
import os, urllib.request
import pandas as pd, numpy as np
import matplotlib.pyplot as plt

# "En vivo o caché": usa el archivo local; si falta (ej. Colab), lo baja del repo.
CSV = "compras_ml.csv"
if not os.path.exists(CSV):
    urllib.request.urlretrieve(f"https://raw.githubusercontent.com/formacionpublica-bootcamp/bootcamp/main/data/compras_ml.csv", CSV)

df = pd.read_csv(CSV)
print("Datos cargados:", df.shape, "filas x columnas")
df.head()

### Datos cargados arriba. Construyamos un dataset limpio y cruzado.

## 1. Cruzar tablas con `merge`

Un dataset analítico suele unir varias fuentes. Derivamos una tabla resumen por categoría y la **cruzamos** de vuelta para enriquecer cada compra con el promedio de su categoría.

### ✍️ Ejercicio 1 — Enriquece cada fila con el promedio de su categoría

In [ ]:
resumen = (df.groupby("categoria")["monto_total"].mean()
             .reset_index().rename(columns={"monto_total": "monto_prom_categoria"}))
# TODO: une df con resumen por la columna "categoria" (how="left")
enriquecido = ...
print(enriquecido[["categoria", "monto_total", "monto_prom_categoria"]].head(3))

In [ ]:
# ── Celda de chequeo · Ejercicio 1 ──────────────────────────────────────────
try:
    assert "monto_prom_categoria" in enriquecido.columns
    assert len(enriquecido) == len(df)
    print("✅ Ejercicio 1: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", "df.merge(resumen, on='categoria', how='left').")
    print("   Detalle:", e)

## 2. Limpiar nulos

Los datos reales tienen huecos. Aquí ensuciamos a propósito y luego decidimos: ¿descartar o imputar?

### ✍️ Ejercicio 2 — Cuenta y elimina los nulos

In [ ]:
sucio = df.copy()
sucio.loc[sucio.sample(50, random_state=1).index, "monto_total"] = np.nan
# TODO: n_nulos = cuántos NaN hay en monto_total ; limpio = df sin esas filas
n_nulos = ...
limpio = ...
print("nulos:", n_nulos, "| filas tras limpiar:", len(limpio))

In [ ]:
# ── Celda de chequeo · Ejercicio 2 ──────────────────────────────────────────
try:
    assert n_nulos == 50
    assert int(limpio["monto_total"].isna().sum()) == 0
    print("✅ Ejercicio 2: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", "sucio['monto_total'].isna().sum() y sucio.dropna(subset=['monto_total']).")
    print("   Detalle:", e)

## 3. Eliminar duplicados

Registros repetidos inflan los conteos. `drop_duplicates` los quita.

### ✍️ Ejercicio 3 — Quita las filas duplicadas

In [ ]:
con_dup = pd.concat([df, df.head(10)], ignore_index=True)
# TODO: elimina las filas duplicadas de con_dup
sin_dup = ...
print("con duplicados:", len(con_dup), "| sin duplicados:", len(sin_dup))

In [ ]:
# ── Celda de chequeo · Ejercicio 3 ──────────────────────────────────────────
try:
    assert len(con_dup) == len(df) + 10
    assert len(sin_dup) == len(df.drop_duplicates())
    print("✅ Ejercicio 3: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'con_dup.drop_duplicates().')
    print("   Detalle:", e)

## 4. Corregir tipos

Una columna con el tipo correcto ahorra memoria y evita errores. Convertimos el tamaño a categórico.

### ✍️ Ejercicio 4 — Convierte tamano_proveedor a category

In [ ]:
df2 = df.copy()
# TODO: convierte la columna tamano_proveedor al tipo "category"
df2["tamano_proveedor"] = ...
es_category = df2["tamano_proveedor"].dtype.name == "category"
print("es category:", es_category)

In [ ]:
# ── Celda de chequeo · Ejercicio 4 ──────────────────────────────────────────
try:
    assert es_category is True
    assert df2["tamano_proveedor"].nunique() == df["tamano_proveedor"].nunique()
    print("✅ Ejercicio 4: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", "df2['tamano_proveedor'].astype('category').")
    print("   Detalle:", e)

In [ ]:
# (ilustración) Promedio por categoría que calculamos al cruzar
r = resumen.sort_values("monto_prom_categoria").tail(8)
plt.figure(figsize=(6, 4))
plt.barh(r["categoria"], r["monto_prom_categoria"], color="#4240e5")
plt.xlabel("Monto promedio"); plt.title("Promedio por categoría (tabla cruzada)")
plt.tight_layout(); plt.show()

## Cierre

- **`merge`** une tablas por una clave; **`groupby`** las resume.
- Decide entre **descartar** o **imputar** los nulos según el caso.
- **`drop_duplicates`** quita repetidos; los **tipos** correctos evitan errores.
- Con un dataset limpio y cruzado, en R2-02 haremos ingeniería de variables.

> *El 80% del trabajo de datos es dejar el dato listo. Hazlo bien y el modelo lo agradece.*